# **<span style="color: black; font-size:1em;">IDAES-LCA Integration (ILI)</span>**

## <span style="color:black; font-weight:bold"> Introduction </span> 

<span style = "color:black">
This document demonstrates the integration of Life Cycle Assessment into the IDAES workflow. The goal is to develop a solution allowing a bi-directional communication between PrOMMiS and openLCA, and enabling a selective, constraint-based, environmental optimization framework 
</br>

<u> Goal and Scope:</u> In this Jupyter Notebook, we use life cycle assessment to evaluate the environmental impact of Rare Earth Oxide Recovery from coal mining refuse.</br>
As shown in the system boundary below, the scope of this work starts with the leaching of size-reduced REE-rich feedstock (REE: Rare Earth Elements) and ends with the recovery of mixed REO solids. The process consists of six main stages: 1) Mixing and Leaching, 2) Rougher Solvent Extraction, 3) Cleaner Solvent Extraction, 4) Precipitation, 5) Solid-Liquid (S/L) separation, and 6) Roasting. </br>
It's crucial to note here that the current script does not account for:

* Upstream processes leading to the production of REE-rich feedstock 


* Downstream processes leading to the separation of REE contained in the REO    

<u> Main Product:</u> The main product of the flowsheet modeled in this Jupyter Notebook is Rare Earth Oxide (REO) solid.

<u> Co-products or by-products:</u> None

<u> Functional Unit:</u> The function unit is 1 kg of recovered REO solids

<u> Allocation:</u> The evaluated flowsheet produces a single product with no intermediate co-products or by-product. As a result, there is no need for allocation in this modeling exercise.

<u> Life cycle inventory Estimation:</u> The material and energy inputs shown in the system boundary figure below have been shortlisted and estimated based on the UKy flowsheet output, [technical reports on the production on the production of REE from coal and coal by-products](https://www.osti.gov/servlets/purl/1569277), and other relevant literature.
</span>

<span style = "color:black">

**Useful notes, definitions, and links:**
- **PrOMMiS model:** Process Optimization and Modeling for Minerals Sustainability initiative, led by the U.S. Department of Energy (DOE), specifically under NETL (National Energy Technology Laboratory). The PrOMMiS source code can be accessed on github through the following repository link: [https://github.com/prommis/prommis/tree/main](https://github.com/prommis/prommis/tree/main)<br> PrOMMiS serves to optimize processing flowsheets. This Jupyter Notebook focuses on a flowsheet for the extraction of REE from coal refuse. 


- **UKy Flowsheet Code:** [https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py](https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py)

</span>

<div style="text-align: center;">
    <img src="images/system_boundary_1.png" width="1000"/>
</div>

## <span style="color:black; font-weight:bold"> Step 1: Import the necessary tools</span> 

In [ ]:
import os

import pandas as pd
from netlolca import NetlOlca
from netlolca import get_logger

import src.foqus_class as foqus_class
from src.foqus_class import NetlFoqus

log = get_logger()

## <span style="color:black; font-weight:bold"> Step 2: Create a New FOQUS Graph, setup output directory, and initiate UKy flowsheet  </span> 

In [ ]:
nf = NetlFoqus()
m = nf.init_uky()

## <span style="color:black; font-weight:bold"> Step 3: FOQUS - Setup the PrOMMiS and openLCA Nodes </span> 

#### <span style="color:black; font-weight:bold"> Select the Decision Variables </span> 

<span style="color:black">
In the following example, two variables are chosen to optimize (i.e., `my_var_1` and `my_var_2`). These are added to the NetlFoqus Python class's list of decision variables (appended to the classes `dv` class attribute) and set as input variables to the PrOMMiS node. The value of each decision variable and their min/max amounts are set using the `initialize_decision_variables` method. For convenicne, these are saved to a CSV file in the output folder defined above.</span>

In [ ]:
# Add a decision variable
my_var1 = "fs.leach_liquid_feed.flow_vol"
nf.add_decision_variable(my_var1)

# [FH] we need at least two decision variables
my_var2 = "fs.load_sep.split_fraction"
nf.add_decision_variable(my_var2)

# Help with initializing decision variables
foqus_class.initialize_decision_variables(nf, m)

# Store decision variable information in a dataframe and save to output directory
dv_df = foqus_class.export_decision_variables(nf)

# set decesion variables as input variables for prommis_node
for dv in nf.dv:
    nf.set_input_variables(nf.prommis_node, dv.ipvname, dv.value, dv.min, dv.max)

#### <span style="color:black; font-weight:bold"> Initialize the intermediate variables </span> 

<span style="color:black">
This step initializes starting values of the exchanges between the PrOMMiS node and the openLCA node and connects the parameters from one node to the other.</span>

In [ ]:
# Initialize intermediate variables
nf.initialize_intermediate_variables(nf.prommis_node, nf.olca_node)

# connect intermediate variables
nf.connect_intermediate_variables(nf.prommis_node, nf.olca_node)

# save exchanges in nf.exchanges
lca_df_finalized = nf.exchanges

#### <span style="color:black; font-weight:bold"> Add additional PrOMMiS model outputs </span> 
<span style="color:black">
Below, a selection of ProMMiS model outputs are added to the model.
These parameters will then be available for optimization and/or contraints to the model as described later on in this notebook.</span> 

In [ ]:
prommis_outputs_df = foqus_class.generate_prommis_outputs(nf, m)
prommis_outputs_df

#### <span style="color:black; font-weight:bold"> Initialize the LCA model </span> 

##### <u><span style="color:black; font-weight:bold">Connect to IPC</span></u>

In [ ]:
netl = NetlOlca()
netl.connect()
netl.read()

##### <u><span style="color:black; font-weight:bold">Enter Process Name and Description</span></u>

In [ ]:
process_name = "TESTING - REO Extraction From Coal Mining Refuse | UKy Flowsheet"

process_description = """This process involves the production of a Rare
Earth Oxide solid extraction from coal mining refuse. The scope of this
work starts with the leaching of size-reduced REE-rich feedstock (REE:
Rare Earth Elements) and ends with the recovery of mixed REO solids. The
process consists of six main stages: 1) Mixing and Leaching, 2) Rougher
Solvent Extraction, 3) Cleaner Solvent Extraction, 4) Precipitation,
5) Solid-Liquid (S/L) separation, and 6) Roasting. This process does not
account for upstream processes leading to the production of REE-rich
feedstock nor does it account for Downstream processes leading to the
separation of REE contained in the REO. The main product is a rare earth
oxide solid with no other by-products or co-products. The functional
unit is 1 kg of recovered REO solids. The material and energy inputs
shown in the system boundary figure below have been shortlisted and
estimated based on the UKy flowsheet output, as well as other relevant
literature."""

##### <u><span style="color:black; font-weight:bold">Select Impact Assessment Method</span></u>

<span style="color:black">
The following cell includes the UUID of the impact assessment method to be used in subsequent steps.
The UUID in the below cell refers to TRACI 2.1 (NETL)</span>

In [ ]:
impact_method_uuid = '60cb71ff-0ef0-4e6c-9ce7-c885d921dd15'

##### <u><span style="color:black; font-weight:bold">Configure Parameter Set</span></u>

In [ ]:
parameter_set_name = "Baseline"
parameter_set_description = "Baseline parameter set for the process"
is_baseline = True

##### <u><span style="color:black; font-weight:bold">Run first step in openLCA</span></u>

<span style="color:black">
The following cell includes the following steps in openLCA:

* Create a new process
* Assign parameters to each exchange
* Create a new product system
* Create a parameter set
* Run analysis using the selected impact caategory
* Export the parameters, impacts, and run information if the save_outputs = True
</span>

In [ ]:
total_impacts, my_parameters, ps_uuid = foqus_class.initiate_lca_model(
    netl,
    process_name,
    process_description,
    lca_df_finalized,
    impact_method_uuid,
    parameter_set_name,
    parameter_set_description,
    is_baseline,
    save_outputs=True,
    output_dir=nf.output_dir
)

##### <u><span style="color:black; font-weight:bold">Create output variables of openLCA node in FOQUS</span></u>
<span style="color:black">
The results from the first openLCA run are used initialize output values from the openLCA node. </br>
This will allow the solver access the openLCA results.</span>

In [ ]:
foqus_class.create_openlca_outputs(nf, total_impacts, nf.olca_node)

##### <u><span style="color:black; font-weight:bold">Define openLCA and PrOMMiS node scripts</span></u>
<span style="color:black">
Below shows the node scripts that were created for running FOQUS. The user should not edit these scripts. </br>
Below a brief description of the functionality of each script.

<u>PrOMMiS node script</u>:
* The PrOMMiS decision variables, also defined as inputs to the PrOMMiS node, are initiated with values in previous steps of this jupyter notebook.
* The initial decision variables values are retrieved and used to build the UKy coal refuse model
* The PrOMMiS flowsheet is evaluated, yielding flosheet design parameters (flow volume, elemental mass concentration, etc.)
* The PrOMMiS raw data is converted to LCA-relevant units and assigned to the respective PrOMMiS node outputs
* The PrOMMiS node outputs are transferred to the openLCA node

<u>openLCA node script</u>:
* The openLCA node inputs are populated with the exchanges' values from the PrOMMiS node
* The exchanges values are retrieved and populated in openLCA to their respective parameters in the pre-defined parameter set.
* The product system is re-evaluated using the updated parameter set.
* The analysis results are exported and assigned to their respective openLCA node output variables

<u>Iteration process</u>:
* The openLCA results (e.g., Global Warming Potential) are later passed to the solver
* Accordingly the solver updates the input decision variables in the PrOMMiS node to be re-evaluated in a subsequent iteration

</span>

In [ ]:
#openLCA node
nf.define_node_script(nf.olca_node, foqus_class.openlca_node_script)

#PrOMMiS node
nf.define_node_script(nf.prommis_node, foqus_class.prommis_node_script)

## <span style="color:black; font-weight:bold"> Step 4: Create Session </span> 
Note that you may see a series of error messages as FOQUS loads optional plugins.
Assuming that NLopt was installed, you may ignore the error messages associated with the other plugins.

In [ ]:
# Enter your foqus working directory
my_session = nf.create_session()

## <span style="color:black; font-weight:bold"> Step 5: Create a problem and setup optimizer </span> 
<span style="color:black">
The first step in setting up the optimizer is defining the problem, which consists of: 

- The working session
- The optimization method ("NLopt")
- The node with the input variables to be optimized</span> 

In [ ]:
problem = nf.setup_optimizer(my_session, "NLopt", nf.prommis_node)

## <span style="color:black; font-weight:bold"> Step 6: Create Problem Objective </span> 

<span style="color:black">

Setting an objective requires a set of inputs including the following:

* The objective expression representing the amount to be minimized
* A penalty scale: This is most needed when there are multiple objectives and at least one constraint. The purpose of the penalty scale is to help the solver apply the constraint violation penalty differently to each different objective function.
* Value for failure:  value to be assigned to the objective function if the objective cannot be evaluated for any reason. The value should be higher than the expected highest value for a successful objective. 

There are different approaches to setting an objective:

* Approach 1: Assigning multiple single objectives. One example could involve creating two separate objectives: 1) Global Warming Potential and 2) Freshwater ecotoxicity. The solution in this case will tend to converge towards minimizing both objectives. A defined penalty scale is generally needed in this case, the following cells include an automated method to generate a penalty factor for each selected objective. 
* Approach 2: Assigning a single objective function that covers multiple variables. This will require the user the input a set of weights for each of the variables to be included in the function - ideally these weights must add up to 1. The developed methods for this approach would automatically apply factors to properly scale all the selected variables. 
One example could invovle the following variables in a single function: Total capital cost with a weight of 0.3, freshwater ecotoxicity with a weight of 0.4, and global warming potential with a weight of 0.3.

Instructions to set an objective:

* Please use one of the next two cells 6.1 and 6.2 to setup your objective. The cells are already setup to depict each of the examples described above
* Variables selection options: Please run the next cell for a list of variables that can be used in the objective setup function. Please ensure the exact same variable names are used in cells 6.1 and 6.2.

</span>

##### <u><span style="color:black; font-weight:bold">Available variables</span></u>

In [ ]:
nf.outVars

##### <u><span style="color:black; font-weight:bold">6.1. Single-variable multiple-objectives setup</span></u>

In [ ]:
objectives_list = ["Freshwater ecotoxicity", "Global Warming Potential [AR6, 100 yr]"]

In [ ]:
ps_guide = foqus_class.generate_penalty_scales(prommis_outputs_df, total_impacts)
penalty_scale_list = foqus_class.get_penalty_scales(objectives_list, ps_guide)

In [ ]:
problem = nf.create_problem_objective_singular(
    problem,
    objectives_list,
    [nf.olca_node, nf.olca_node],
    [10000000, 1000000], #dummy values
    penalty_scale_list
)

##### <u><span style="color:black; font-weight:bold">6.2. Multi-variable single objectives setup</span></u>

In [ ]:
objectives_list = ["total plant cost", "Freshwater ecotoxicity", "Global Warming Potential [AR6, 100 yr]"]

In [ ]:
ps_guide = foqus_class.generate_penalty_scales(
    prommis_outputs_df,
    total_impacts
)
penalty_scale_list = foqus_class.get_penalty_scales(objectives_list, ps_guide)

In [ ]:
problem = nf.create_problem_objective_multiple(
    problem,
    objectives_list,
    [nf.prommis_node, nf.olca_node, nf.olca_node],
    [1.3, 10000000, 1000000], #dummy values
    penalty_scale_list,
    [0.3, 0.4, 0.3]
)

## <span style="color:black; font-weight:bold"> Step 7: Create Problem Constraint <i>(Optional)</i> </span> 

<span style="color:black">

Backroung, context, and instructions to setup a problem constraint:

* Contraint contextualization: Setting up a constraint helps exclude potential solutions that meet the objective but violate certain conditions such as the operating cost or the REE recovery rate. 

* The user has to define an inquality contraint function g(x) of the form:
    g(x) < 0

* If the constraint is violated, the solver generates a penalty that is applied to the objective. This penalty depends on two user inputs: 1) penalty factor and 2) penalty form. The effect of the penalty factor and penalty form can be show in the Equations (1-3) below. <i><b><u>Source</u></b>: [FOQUS Documentation - Optimization](https://foqus.readthedocs.io/en/stable/chapt_opt/reference/optimization.html)</i>

* <b><u>Example</b></u>: Contraint to set a minimum recovery rate of 90%. In this case the constraint function g(x) = 0.9 - (Recovery Rate).

* Instructions:
    * If the constraint consists of setting a maximum value for a variable, please use <b>nf.get_max_constraint</b> under section 7.1
    * If the constraint consists of setting a minimum value for a variable, please use <b>nf.get_min_constraint</b> under section 7.1
    * After creating the constraint function, proceed to section 7.2 in order to assign the constraint to the problem

<span style = "color:red"><b><u>Important Note:</b></u></span> 

* Outputs used to define decision variables or problem objectives cannot be used as constraints.
* If you want to set more than one constraint (for example a 2nd constraint), create a new function "g_x_2" and assign it to the problem using the "create_problem_constraint" method, which will append the 2nd constraint and not overwrite any previosuly set constraints.
* Run the cell under 'Available Variables' to get a list of variables with their respective nodes, that can be used to create a constraint

</span>

<div style="text-align: center;">
    <img src="images/penalty_forms.png" width="650"/>
</div>

##### <u><span style="color:black; font-weight:bold">Available variables</span></u>

In [ ]:
nf.outVars

##### <u><span style="color:black; font-weight:bold">7.1. Generate constraint function </span></u>
_Choose one of the following examples for `g(x)`._

##### <u><span style="color:black; font-weight:bold">Generating constraint to set maximum variable value</span></u>

In [ ]:
g_x = nf.get_max_constraint(variable="Water Consumption (NETL)",
                            node=nf.olca_node,
                            max_value=1000
                            )

##### <u><span style="color:black; font-weight:bold">Generating constraint to set minimum variable value</span></u>

In [ ]:
g_x = nf.get_min_constraint(variable="recovery rate",
                            node=nf.prommis_node,
                            min_value=0.2
                            )

##### <u><span style="color:black; font-weight:bold">7.2. Assign constraint to the problem </span></u>

In [ ]:
problem = nf.create_problem_constraint (problem,
                                        pycode=g_x,
                                        penalty_factor=10,
                                        form='Linear'
                                        )

## <span style="color:black; font-weight:bold"> Step 8: Setup solver options </span> 

In [ ]:
problem = nf.setup_nlopt_solver_options(problem, True)

## <span style="color:black; font-weight:bold"> Step 9: Run Optimization </span> 
_Note that optimization may take 10 minutes or more to complete._

In [ ]:
my_solver, problem = nf.run_optimization(problem, my_session)

## <span style="color:black; font-weight:bold"> Step 10: Extract Results </span> 

<span style="color:black"> The following cell extracts the optimization results and updates the parameter set of the product system in openLCA to reflect the parameters corresponding to the optimized solution.</span>

In [ ]:
parameters_s, impacts_s, dv_s, prommis_s = foqus_class.get_optimization_results(
    client=netl,
    ps_uuid=ps_uuid,
    parameter_set_name=parameter_set_name,
    solver=my_solver,
    decision_variables=pd.read_csv(os.path.join(nf.output_dir, "decision_variables.csv")),
    prommis_outputs=pd.read_csv(os.path.join(nf.output_dir, "prommis_outputs.csv")),
    parameters=pd.read_csv(os.path.join(nf.output_dir, "my_parameters.csv")),
    total_impacts=pd.read_csv(os.path.join(nf.output_dir, "total_impacts.csv"))
)

##### <u><span style="color:black; font-weight:bold">Final Parameters</span></u>

<span style="color:black">The following cell prints out the final exchanges corresponding to the optimal solution.</span>

In [ ]:
parameters_s

##### <u><span style="color:black; font-weight:bold">Final Environmental Impacts</span></u>

<span style="color:black">The following cell prints out the environmental impacts corresponding to the optimal solution.</span>

In [ ]:
impacts_s

##### <u><span style="color:black; font-weight:bold">Optimal Values of Decision Variables</span></u>

<span style="color:black">The following cell prints out the decision variables values corresponding to the optimal solution.</span>

In [ ]:
dv_s

##### <u><span style="color:black; font-weight:bold">Final PrOMMiS Model Outputs</span></u>

<span style="color:black">The following cell prints out the PrOMMiS flowsheet design outputs corresponding to the optimal solution.</span>

In [ ]:
prommis_s